# Phase 7 - Prior Case Retrieval (lightweight Phase-1 model)

**Task:** for each query case, rank the cases of its candidate pool by legal relevance; exactly 7 unique candidate ids per query are submitted. Primary metric micro-F1@7, secondary micro-precision@7 and micro-recall@7 (against the full gold set).

**What you need on disk (nothing else - the model code is embedded below):**
1. your Phase-1 lightweight checkpoint `lightweight_final.pt` (must contain `model_state` and `config`)
2. the Phase-1 tokenizer pickle `legal_bpe_v3.pkl` (the `.pt` does **not** contain it)
3. a Hugging Face token (Kaggle secret `hf_token`)

**Approach (dual encoder):** a long case is cut into `CH`-token chunks, `K_CHUNKS` of them are chosen (`spread` = evenly over the whole case); the Phase-1 lightweight model (4 shared blocks run twice, sliding-window attention) reads every chunk; pooled state -> linear projection -> chunk vector; the case vector is the mean of its chunk vectors; relevance is cosine similarity. Training is contrastive (InfoNCE) with in-batch negatives plus **tf-idf hard negatives**; relevant cases are never used as negatives. At inference two similarity functions are compared: the case-vector cosine and a chunk-level **max-sim**; the model score can optionally be fused with a tf-idf score (tuned on dev - reported as post-processing).

**Experiments:** zero-shot Phase-1 features and a tf-idf baseline (for the transfer analysis), then full fine-tuning, partial, LoRA, **LoRMA+** (compulsory), Houlsby bottleneck adapters, prompt tuning, knowledge distillation (student = the 2 of 4 unique blocks 0 and 2, still run twice; trained on teacher similarity scores), plus pooling / loss / hard-negative / similarity ablations and a score ensemble. AdaLoRA, QLoRA, prefix tuning and IA3 are **not** included.

> The course requires >= 6 substantially different experiments (>= 5 different adaptation techniques, LoRMA and knowledge distillation among them). The budgets (`minutes=`), learning rates, temperatures and batch sizes are placeholders - choosing and justifying them is your job (course rules). Training is time-budgeted: check the `epochs` figure printed after each run.

## 0. Config - edit the paths

In [ ]:
TOKENIZER_PATH = "/kaggle/input/YOUR_DATASET/legal_bpe_v3.pkl"      # <-- Phase-1 tokenizer
CKPT_PATH      = "/kaggle/input/YOUR_DATASET/lightweight_final.pt"       # <-- Phase-1 lightweight checkpoint
HF_REPO   = "Exploration-Lab/CS-781-Capstone"
PHASE_DIR = "Phase_7_PCR"
OUT = "/kaggle/working"

CH         = 512        # tokens per chunk (<= the model's max_seq_len of 1024)
K_CHUNKS   = 4          # chunks per document used at inference (cost is linear in this)
SELECT     = "spread"   # which chunks of a long case: head | tail | headtail | spread (evenly spaced over the whole case)
CAP_CHARS  = 12000      # characters read from each end of a case (bounds tokenisation time)
TOPK       = 7          # the evaluator uses the top 7 candidates per query
EMB_DIM    = 256        # size of the retrieval embedding (linear projection of the pooled hidden state)
TAU        = 0.05       # softmax temperature of the contrastive loss
HARD_POOL  = 30         # hard negatives are sampled from the top-HARD_POOL tf-idf neighbours of a query (relevant cases excluded)

## 1. Imports and Hugging Face login

In [ ]:
import copy, json, math, os, pickle, random, re, time, zipfile
from collections import defaultdict
import numpy as np
import torch, torch.nn as nn, torch.nn.functional as F
from huggingface_hub import hf_hub_download

dev = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", dev, "| torch", torch.__version__)

from collections import Counter

In [ ]:
try:
    from kaggle_secrets import UserSecretsClient
    HF_TOKEN = UserSecretsClient().get_secret("hf_token")
except Exception:
    HF_TOKEN = os.environ.get("HF_TOKEN")        # Colab / local: export HF_TOKEN yourself
from huggingface_hub import login
if HF_TOKEN:
    login(token=HF_TOKEN); print("HF login ok")
else:
    print("no HF token found - dataset download may fail")

## 2. Phase-1 tokenizer (class copied from your Phase-1 notebook)

In [ ]:
class LegalBPETokenizer:
    """Phase-1 tokenizer, copied unchanged from your Phase-1 notebook (needs legal_bpe_v3.pkl)."""

    def __init__(self, model_path):
        with open(model_path, "rb") as f:
            model = pickle.load(f)
        self.vocab_size = model["vocab_size"]
        self.merges = model["merges"]                     # [((a, b), new_id), ...]
        self.lowercase = model.get("lowercase", False)
        self.special_tokens = model.get("special_tokens", [])
        self.pattern = re.compile(model["pattern"])
        self.merge_rank = {tuple(pair): rank for rank, (pair, _) in enumerate(self.merges)}
        self.token_bytes = {i: bytes([i]) for i in range(256)}
        for (l, r), new_id in self.merges:
            self.token_bytes[new_id] = self.token_bytes[l] + self.token_bytes[r]
        base = 256 + len(self.merges)
        self.special_to_id = {t: base + i for i, t in enumerate(self.special_tokens)}
        self.id_to_special = {v: k for k, v in self.special_to_id.items()}
        self.eos_id = self.special_to_id.get("<|endoftext|>")
        self._special_re = (re.compile("(" + "|".join(re.escape(t) for t in self.special_tokens) + ")")
                            if self.special_tokens else None)
        self._cache = {}

    def _encode_piece(self, piece_bytes):
        cached = self._cache.get(piece_bytes)
        if cached is not None:
            return cached
        ids = list(piece_bytes)
        while len(ids) > 1:
            best, best_rank = None, None
            for p in zip(ids, ids[1:]):
                r = self.merge_rank.get(p)
                if r is not None and (best_rank is None or r < best_rank):
                    best, best_rank = p, r
            if best is None:
                break
            new_id = 256 + best_rank
            out, i = [], 0
            while i < len(ids):
                if i < len(ids) - 1 and ids[i] == best[0] and ids[i + 1] == best[1]:
                    out.append(new_id); i += 2
                else:
                    out.append(ids[i]); i += 1
            ids = out
        if len(self._cache) < 500_000:
            self._cache[piece_bytes] = ids
        return ids

    def _encode_ordinary(self, text):
        if self.lowercase:
            text = text.lower()
        out = []
        for piece in self.pattern.findall(text):
            out.extend(self._encode_piece(piece.encode("utf-8")))
        return out

    def encode(self, text, add_eos=False):
        if self._special_re is not None:
            ids = []
            for part in self._special_re.split(text):
                if part in self.special_to_id:
                    ids.append(self.special_to_id[part])
                elif part:
                    ids.extend(self._encode_ordinary(part))
        else:
            ids = self._encode_ordinary(text)
        if add_eos:
            ids.append(self.eos_id)
        return ids

    def decode(self, ids):
        chunks = []
        for token_id in ids:
            if token_id in self.id_to_special:
                chunks.append(self.id_to_special[token_id].encode("utf-8"))
            else:
                chunks.append(self.token_bytes[token_id])
        return b"".join(chunks).decode("utf-8", errors="replace")


tokenizer = LegalBPETokenizer(TOKENIZER_PATH)
print("vocab size:", tokenizer.vocab_size, "| lowercase:", tokenizer.lowercase, "| eos id:", tokenizer.eos_id)
print([tokenizer.decode([i]) for i in tokenizer.encode("The appellant filed this appeal.")])

## 3. Phase-1 lightweight model code (copied unchanged) and checkpoint loading

In [ ]:

# ---- lightweight_layers.py (Phase 1, unchanged) ----
class RMSNorm(nn.Module):
    def __init__(self, dim, eps=1e-6):
        super().__init__()
        self.weight = nn.Parameter(torch.ones(dim))
        self.eps = eps
    def forward(self, x):
        rms = x.float().pow(2).mean(dim=-1, keepdim=True)
        x = x * torch.rsqrt(rms + self.eps)
        return x * self.weight.to(x.dtype)


def get_alibi_slopes(n_heads):
    def power_of_2_slopes(n):
        start = 2 ** (-2 ** -(math.log2(n) - 3))
        ratio = start
        return torch.tensor([start * ratio ** i for i in range(n)], dtype=torch.float32)
    if math.log2(n_heads).is_integer():
        return power_of_2_slopes(n_heads)
    closest_power = 2 ** int(math.floor(math.log2(n_heads)))
    extra = n_heads - closest_power
    slopes = power_of_2_slopes(closest_power)
    extra_slopes = get_alibi_slopes(2 * closest_power)[0::2]
    return torch.cat([slopes, extra_slopes[:extra]])


class SwiGLU(nn.Module):
    def __init__(self, d_model, d_ff, dropout):
        super().__init__()
        self.w1 = nn.Linear(d_model, d_ff, bias=False)
        self.w2 = nn.Linear(d_model, d_ff, bias=False)
        self.w3 = nn.Linear(d_ff, d_model, bias=False)
        self.dropout = nn.Dropout(dropout)
    def forward(self, x):
        return self.dropout(self.w3(F.silu(self.w1(x)) * self.w2(x)))


class LocalGQAAttention(nn.Module):
    """Causal sliding-window grouped-query attention with ALiBi."""
    def __init__(self, d_model, n_q_heads, n_kv_heads, max_seq_len, window_size, dropout, qk_norm=True):
        super().__init__()
        assert d_model % n_q_heads == 0 and n_q_heads % n_kv_heads == 0
        self.d_model = d_model
        self.n_q_heads, self.n_kv_heads = n_q_heads, n_kv_heads
        self.groups = n_q_heads // n_kv_heads
        self.d_head = d_model // n_q_heads
        self.window_size = min(window_size, max_seq_len)
        self.q_proj = nn.Linear(d_model, d_model, bias=False)
        self.k_proj = nn.Linear(d_model, n_kv_heads * self.d_head, bias=False)
        self.v_proj = nn.Linear(d_model, n_kv_heads * self.d_head, bias=False)
        self.out_proj = nn.Linear(d_model, d_model, bias=False)
        self.qk_norm = qk_norm
        if qk_norm:
            self.q_norm = RMSNorm(self.d_head)
            self.k_norm = RMSNorm(self.d_head)
        self.dropout = dropout
        self.register_buffer("alibi_slopes", get_alibi_slopes(n_q_heads), persistent=False)
        distance = torch.arange(self.window_size - 1, -1, -1, dtype=torch.float32)
        self.register_buffer("distance", distance, persistent=False)
        query_pos = torch.arange(max_seq_len).unsqueeze(1)
        window_pos = torch.arange(self.window_size).unsqueeze(0)
        valid = window_pos >= (self.window_size - 1 - query_pos)
        self.register_buffer("valid_mask", valid, persistent=False)

    def forward(self, x):
        B, T, C = x.shape
        q = self.q_proj(x).view(B, T, self.n_q_heads, self.d_head).transpose(1, 2)
        k = self.k_proj(x).view(B, T, self.n_kv_heads, self.d_head).transpose(1, 2)
        v = self.v_proj(x).view(B, T, self.n_kv_heads, self.d_head).transpose(1, 2)
        if self.qk_norm:
            q, k = self.q_norm(q), self.k_norm(k)
        q = q.view(B, self.n_kv_heads, self.groups, T, self.d_head)
        W = self.window_size
        k_win = F.pad(k, (0, 0, W - 1, 0)).unfold(2, W, 1).permute(0, 1, 2, 4, 3)   # (B, Hkv, T, W, D)
        v_win = F.pad(v, (0, 0, W - 1, 0)).unfold(2, W, 1).permute(0, 1, 2, 4, 3)
        scores = torch.einsum("bhgtd,bhtwd->bhgtw", q, k_win) / math.sqrt(self.d_head)
        slopes = self.alibi_slopes.view(1, self.n_kv_heads, self.groups, 1, 1)
        scores = scores - slopes * self.distance.view(1, 1, 1, 1, W)
        mask = self.valid_mask[:T, :W].view(1, 1, 1, T, W)
        scores = scores.masked_fill(~mask, float("-inf"))
        attn = F.softmax(scores.float(), dim=-1).to(scores.dtype)
        if self.training and self.dropout > 0:
            attn = F.dropout(attn, p=self.dropout)
        out = torch.einsum("bhgtw,bhtwd->bhgtd", attn, v_win)
        out = out.reshape(B, self.n_q_heads, T, self.d_head).transpose(1, 2).contiguous().view(B, T, C)
        return self.out_proj(out)


class LightweightBlock(nn.Module):
    def __init__(self, d_model, n_q_heads, n_kv_heads, max_seq_len, window_size, d_ff, dropout, qk_norm):
        super().__init__()
        self.ln1 = RMSNorm(d_model)
        self.attn = LocalGQAAttention(d_model, n_q_heads, n_kv_heads, max_seq_len, window_size, dropout, qk_norm)
        self.ln2 = RMSNorm(d_model)
        self.ffn = SwiGLU(d_model, d_ff, dropout)
    def forward(self, x):
        x = x + self.attn(self.ln1(x))
        return x + self.ffn(self.ln2(x))


# ---- model_arch_lightweight.py (Phase 1, unchanged) ----
from dataclasses import dataclass

@dataclass
class LightweightConfig:
    vocab_size: int
    d_model: int = 1024
    n_q_heads: int = 16
    n_kv_heads: int = 4
    d_ff: int = 4096
    max_seq_len: int = 1024
    window_size: int = 128
    dropout: float = 0.0
    qk_norm: bool = True
    n_unique_blocks: int = 4       # 4 unique blocks ...
    n_repeats: int = 2             # ... each executed twice: B1 B2 B3 B4 B1 B2 B3 B4


class LightweightLM(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.cfg = cfg
        self.tok_emb = nn.Embedding(cfg.vocab_size, cfg.d_model)
        self.blocks = nn.ModuleList([
            LightweightBlock(cfg.d_model, cfg.n_q_heads, cfg.n_kv_heads, cfg.max_seq_len, cfg.window_size,
                             cfg.d_ff, cfg.dropout, cfg.qk_norm)
            for _ in range(cfg.n_unique_blocks)])
        self.ln_f = RMSNorm(cfg.d_model)
        self.lm_head = nn.Linear(cfg.d_model, cfg.vocab_size, bias=False)
        self.lm_head.weight = self.tok_emb.weight            # weight tying
        self.apply(self._init_weights)

    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def forward(self, x):
        x = self.tok_emb(x)
        for _ in range(self.cfg.n_repeats):
            for block in self.blocks:
                x = block(x)
        return self.lm_head(self.ln_f(x))


In [ ]:

def load_lm(path=CKPT_PATH, dropout=0.1):
    # weights_only=False: this is YOUR OWN checkpoint (config dict + tensors), not an untrusted file
    ck = torch.load(path, map_location="cpu", weights_only=False)
    cfg = LightweightConfig(**{k: v for k, v in ck["config"].items() if k in LightweightConfig.__dataclass_fields__})
    assert cfg.vocab_size == tokenizer.vocab_size, \
        f"checkpoint vocab {cfg.vocab_size} != tokenizer vocab {tokenizer.vocab_size}: wrong tokenizer .pkl?"
    cfg.dropout = dropout                                    # Phase 1 used 0.0; a little helps fine-tuning
    lm = LightweightLM(cfg)
    lm.load_state_dict(ck["model_state"])                    # alibi / mask buffers are non-persistent: rebuilt, not loaded
    return lm

_lm = load_lm(dropout=0.0)
print(_lm.cfg)
print(f"loaded Phase-1 lightweight LM: {sum(p.numel() for p in _lm.parameters())/1e6:.1f}M unique params "
      f"({_lm.cfg.n_unique_blocks} blocks x {_lm.cfg.n_repeats} repeats)")
assert _lm.cfg.max_seq_len >= CH, "CH must be <= max_seq_len"


## 4. Data\n### 4.1 Peek at the raw JSON (verify the layout before trusting the parser)

In [ ]:
from huggingface_hub import list_repo_files
files = sorted(f for f in list_repo_files(HF_REPO, repo_type="dataset") if f.startswith(PHASE_DIR))
print("\n".join(files))

def hf_json(path):
    return json.load(open(hf_hub_download(HF_REPO, path, repo_type="dataset"), encoding="utf-8"))

for name, path in [("train queries", "Train/train.json"), ("train candidates", "Train/train_candidates.json")]:
    raw = hf_json(f"{PHASE_DIR}/{path}")
    k, v = next(iter(raw.items()))
    print(name, type(raw).__name__, "len", len(raw), "| first id:", k, "| keys:", list(v.keys()) if isinstance(v, dict) else type(v).__name__,
          "| text type:", type(v["text"]).__name__ if isinstance(v, dict) else "", "| sentences:", len(v["text"]) if isinstance(v, dict) and isinstance(v["text"], list) else "")
    if "relevant_candidates" in v: print("   relevant_candidates (first query):", v["relevant_candidates"][:8])
del raw

### 4.2 Load train / dev / test (queries + candidate pools)

In [ ]:
def _txt(v):
    t = v["text"] if isinstance(v, dict) else v
    return " ".join(str(s) for s in t) if isinstance(t, (list, tuple)) else str(t)

def load_split(split, qfile, cfile):
    Q = hf_json(f"{PHASE_DIR}/{split}/{qfile}"); C = hf_json(f"{PHASE_DIR}/{split}/{cfile}")
    cands = [dict(id=str(k), text=_txt(v)) for k, v in C.items()]
    cid = {c["id"]: i for i, c in enumerate(cands)}
    queries, missing = [], 0
    for k, v in Q.items():
        rel = [str(r) for r in (v.get("relevant_candidates", []) if isinstance(v, dict) else [])]
        miss = [r for r in rel if r not in cid]; missing += len(miss)
        queries.append(dict(id=str(k), text=_txt(v), rel=[r for r in rel if r in cid], rel_idx=[cid[r] for r in rel if r in cid], n_gold=len(rel)))
    return queries, cands, cid, missing

tr_q, tr_c, tr_cid, m1 = load_split("Train", "train.json", "train_candidates.json")
dv_q, dv_c, dv_cid, m2 = load_split("Dev", "dev.json", "dev_candidates.json")
raw_te, te_c_raw = hf_json(f"{PHASE_DIR}/Test/test.json"), hf_json(f"{PHASE_DIR}/Test/test_candidates.json")
te_q = [dict(id=str(k), text=_txt(v)) for k, v in raw_te.items()]
te_c = [dict(id=str(k), text=_txt(v)) for k, v in te_c_raw.items()]
del raw_te, te_c_raw

for name, qs, cs, miss in [("train", tr_q, tr_c, m1), ("dev", dv_q, dv_c, m2)]:
    ng = [q["n_gold"] for q in qs]
    print(f"{name}: {len(qs)} queries | pool {len(cs)} candidates | relevant per query: mean {np.mean(ng):.1f}, min {min(ng)}, max {max(ng)} "
          f"| relevant ids missing from the pool: {miss} | median chars: query {int(np.median([len(q['text']) for q in qs]))}, candidate {int(np.median([len(c['text']) for c in cs]))}")
print(f"test: {len(te_q)} queries | pool {len(te_c)} candidates")
tr_q = [q for q in tr_q if q["rel_idx"]]                       # training queries need at least one relevant case in the pool
print("training queries with >= 1 relevant case in the pool:", len(tr_q))
print("query ids that also occur in their own pool (excluded from ranking): train", sum(q["id"] in tr_cid for q in tr_q),
      "| dev", sum(q["id"] in dv_cid for q in dv_q), "| test", sum(q["id"] in {c["id"] for c in te_c} for q in te_q))

### 4.3 Chunking and chunk selection

In [ ]:
def select_idx(n, select=None, K=None):
    """Which of the n chunks of a case are used.  head = first K, tail = last K, headtail = first K/2 + last K/2,
    spread = K evenly spaced chunks over the whole case."""
    select, K = select or SELECT, K or K_CHUNKS
    if n <= K:
        return list(range(n))
    if select == "head":
        return list(range(K))
    if select == "tail":
        return list(range(n - K, n))
    if select == "headtail":
        h = K // 2
        return list(range(h)) + list(range(n - (K - h), n))
    if select == "spread":
        return sorted({round(i * (n - 1) / max(1, K - 1)) for i in range(K)})
    raise ValueError(select)

EOS = tokenizer.eos_id

def doc_all_chunks(d):
    """All CH-token chunks of a case (cached): the first and last CAP_CHARS characters are tokenised and cut into chunks."""
    if "_ch" not in d:
        t = d["text"]
        if len(t) > 2 * CAP_CHARS:
            t = t[:CAP_CHARS] + " " + t[-CAP_CHARS:]
        ids = tokenizer.encode(t) or [EOS]
        ch = [ids[s:s + CH] for s in range(0, len(ids), CH)]
        if len(ch) > 1 and len(ch[-1]) < 32:
            ch.pop()
        d["_ch"] = ch
    return d["_ch"]

def doc_chunks(d, K=None, select=None):
    ch = doc_all_chunks(d)
    return [ch[i] for i in select_idx(len(ch), select, K)]

def pad_batch(seqs):
    X = torch.full((len(seqs), max(len(s) for s in seqs)), EOS, dtype=torch.long)
    for i, s in enumerate(seqs):
        X[i, :len(s)] = torch.tensor(s)
    return X, torch.tensor([len(s) for s in seqs])

t0 = time.time()
for d in tr_c[:30]: doc_all_chunks(d)
print(f"tokenisation: {(time.time()-t0)/30*1000:.0f} ms per case | chunks per case (first 30): median "
      f"{int(np.median([len(d['_ch']) for d in tr_c[:30]]))}, max {max(len(d['_ch']) for d in tr_c[:30])}")
print("example chunk selections for a 12-chunk case:", {s: select_idx(12, s, 4) for s in ("head", "tail", "headtail", "spread")})

### 4.4 Checkpoint / tokenizer sanity check

In [ ]:
# Does checkpoint + tokenizer agree? LM loss on real legal text should be ~3-4 nats; ~10 means a wrong tokenizer pkl.
x = torch.tensor([doc_all_chunks(tr_c[0])[0]]); _lm.eval()
with torch.no_grad():
    lg = _lm(x)
print("Phase-1 LM loss on a candidate chunk:", round(F.cross_entropy(lg[0, :-1], x[0, 1:]).item(), 3), "| tokens:", x.size(1))
del _lm

## 5. Metric (micro-F1/P/R @7), ranking, score fusion

In [ ]:
def micro_prf(pred, gold, k=TOPK):
    """Micro precision / recall / F1 @k. pred: {qid: ranked candidate ids}; gold: {qid: set of relevant ids (the full set, not truncated)}."""
    tp = n_pred = n_gold = 0
    for q, g in gold.items():
        top = list(pred.get(q, []))[:k]
        tp += len(set(top) & set(g)); n_pred += len(top); n_gold += len(set(g))
    p = tp / n_pred if n_pred else 0.0; r = tp / n_gold if n_gold else 0.0
    return dict(f1=2 * p * r / (p + r) if p + r else 0.0, p=p, r=r)

def self_mask(queries, cands):
    ids = {c["id"]: i for i, c in enumerate(cands)}
    m = np.zeros((len(queries), len(cands)), dtype=bool)
    for qi, q in enumerate(queries):
        if q["id"] in ids: m[qi, ids[q["id"]]] = True
    return m

def rank_top(S, queries, cands, k=TOPK):
    """score matrix (queries x candidates) -> {qid: top-k candidate ids}; a query never retrieves itself."""
    S = np.where(self_mask(queries, cands), -np.inf, S)
    top = np.argsort(-S, axis=1)[:, :k]
    return {q["id"]: [cands[j]["id"] for j in top[i]] for i, q in enumerate(queries)}

def evaluate_scores(S, queries, cands, name="", verbose=True):
    gold = {q["id"]: set(q["rel"]) for q in queries}
    r = micro_prf(rank_top(S, queries, cands), gold)
    if verbose: print(f"   {name:34s} micro-F1@7 {r['f1']:.4f} | P@7 {r['p']:.4f} | R@7 {r['r']:.4f}")
    return r

def zrow(S):
    return (S - S.mean(1, keepdims=True)) / (S.std(1, keepdims=True) + 1e-8)

def fuse(S_model, S_tfidf, w):
    """Rank fusion of the model score and the tf-idf score (row z-scores): w * model + (1 - w) * tf-idf."""
    return w * zrow(S_model) + (1 - w) * zrow(S_tfidf)

## 6. Lexical baseline (tf-idf) and hard-negative mining\nNo neural model here: this is the reference point for the transfer analysis and the source of the hard negatives.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

def tfidf_scores(queries, cands):
    """Classic lexical baseline (no neural model): cosine of sublinear-tf-idf vectors, vocabulary fitted on the candidate pool."""
    vec = TfidfVectorizer(sublinear_tf=True, min_df=2, max_df=0.6, token_pattern=r"[a-z]{3,}", lowercase=True, dtype=np.float32)
    X = vec.fit_transform([c["text"][:100000] for c in cands])
    Qm = vec.transform([q["text"][:100000] for q in queries])
    return (Qm @ X.T).toarray()

t0 = time.time()
S_tf_tr, S_tf_dv, S_tf_te = tfidf_scores(tr_q, tr_c), tfidf_scores(dv_q, dv_c), tfidf_scores(te_q, te_c)
print(f"tf-idf scores computed in {time.time()-t0:.0f}s")
res_tfidf = evaluate_scores(S_tf_dv, dv_q, dv_c, "tf-idf baseline (dev)")

# hard negatives for contrastive training: the highest tf-idf neighbours of a query that are NOT relevant (and not the query itself)
sm = self_mask(tr_q, tr_c)
for i, q in enumerate(tr_q):
    s = np.where(sm[i], -np.inf, S_tf_tr[i]); rel = set(q["rel_idx"])
    q["hard"] = [int(j) for j in np.argsort(-s) if int(j) not in rel][:HARD_POOL]
print("hard negatives per training query:", HARD_POOL, "| share of tf-idf top-7 that is relevant (train):",
      round(float(np.mean([len(set(np.argsort(-S_tf_tr[i])[:7]) & set(q['rel_idx'])) / 7 for i, q in enumerate(tr_q)])), 3))

## 7. Adaptation methods: LoRA, LoRMA+, bottleneck adapters

In [ ]:
class LoRALinear(nn.Module):
    """LoRA: W x + s * B A x with W frozen.  rs=True -> rank-stabilised scaling s = alpha / sqrt(r) (rsLoRA)."""
    def __init__(self, base, r=16, alpha=32, rs=False, dropout=0.05):
        super().__init__()
        self.base = base
        self.A = nn.Parameter(torch.empty(r, base.in_features)); nn.init.kaiming_uniform_(self.A, a=math.sqrt(5))
        self.B = nn.Parameter(torch.zeros(base.out_features, r))
        self.scale = alpha / math.sqrt(r) if rs else alpha / r
        self.drop = nn.Dropout(dropout)
    def forward(self, x):
        return self.base(x) + (self.drop(x) @ self.A.T @ self.B.T) * self.scale

def apply_lora(module, r=16, alpha=32, rs=False):
    for name, child in list(module.named_children()):
        if isinstance(child, nn.Linear):
            setattr(module, name, LoRALinear(child, r, alpha, rs))
        else:
            apply_lora(child, r, alpha, rs)


class LoRMALinear(nn.Module):
    """LoRMA+ (Bihany, Patel, Modi 2025): multiplicative update  h = (I + (alpha/r) B A) W0 x ,  B=0 at init."""
    def __init__(self, base, r=8, alpha=8):
        super().__init__()
        self.base = base
        d = base.out_features
        self.A = nn.Parameter(torch.empty(r, d)); nn.init.kaiming_uniform_(self.A, a=math.sqrt(5))
        self.B = nn.Parameter(torch.zeros(d, r))
        self.scale = alpha / r
    def forward(self, x):
        z = F.linear(x, self.base.weight)
        z = z + self.scale * ((z @ self.A.T) @ self.B.T)
        return z + self.base.bias if self.base.bias is not None else z

def apply_lorma(module, targets=("q_proj", "v_proj"), r=8, alpha=8):
    for name, child in list(module.named_children()):
        if isinstance(child, nn.Linear) and name in targets:
            setattr(module, name, LoRMALinear(child, r, alpha))
        else:
            apply_lorma(child, targets, r, alpha)


class IA3Out(nn.Module):
    """IA3: learned vector rescales the layer OUTPUT (used for keys and values). Init = ones (identity)."""
    def __init__(self, base):
        super().__init__(); self.base = base; self.l = nn.Parameter(torch.ones(base.out_features))
    def forward(self, x):
        return self.base(x) * self.l

class IA3In(nn.Module):
    """IA3 on the FFN: learned vector rescales the hidden activation that feeds the second FFN linear."""
    def __init__(self, base):
        super().__init__(); self.base = base; self.l = nn.Parameter(torch.ones(base.in_features))
    def forward(self, x):
        return self.base(x * self.l)

def apply_ia3(blocks):
    for b in blocks:
        b.attn.k_proj = IA3Out(b.attn.k_proj)
        b.attn.v_proj = IA3Out(b.attn.v_proj)
        b.ffn.net[2] = IA3In(b.ffn.net[2])

In [ ]:
class SubAdapter(nn.Module):
    """Bottleneck adapter on a sub-layer OUTPUT, before the residual add: y + up(GELU(down(y))). `up` starts at zero = identity."""
    def __init__(self, sub, d_model, bottleneck=64):
        super().__init__()
        self.sub = sub
        self.down = nn.Linear(d_model, bottleneck); self.up = nn.Linear(bottleneck, d_model)
        nn.init.zeros_(self.up.weight); nn.init.zeros_(self.up.bias)
    def forward(self, x):
        y = self.sub(x)
        return y + self.up(F.gelu(self.down(y)))

def apply_adapters(lm, bottleneck=64):
    """Pfeiffer-style: one bottleneck adapter after the FFN of every one of the 12 blocks (both GRU and attention blocks have an FFN)."""
    for b in lm.blocks:
        b.ffn = SubAdapter(b.ffn, lm.cfg.d_model, bottleneck)

## 8. Dual-encoder model, `build_model`, distillation student (memory-safe: gradient checkpointing, B=2)

In [ ]:
from torch.utils.checkpoint import checkpoint

class Retriever(nn.Module):
    """Dual encoder. Phase-1 lightweight model reads a CH-token chunk -> pooled state (mean | last | meanlast) -> linear projection -> chunk vector.
    A case vector is the mean of its chunk vectors, L2-normalised; relevance = cosine similarity (temperature TAU in the loss)."""
    def __init__(self, lm, pool="mean", emb_dim=EMB_DIM, n_prompt=0, dropout=0.1):
        super().__init__()
        self.lm, self.pool, self.n_prompt = lm, pool, n_prompt
        d = lm.cfg.d_model
        self.proj = nn.Sequential(nn.Dropout(dropout), nn.Linear(2 * d if pool == "meanlast" else d, emb_dim))
        self.prompt = nn.Parameter(torch.randn(n_prompt, d) * 0.02) if n_prompt else None
        self.raw, self.n_nograd, self.use_ckpt = False, 0, False        # raw=True: skip the projection (zero-shot Phase-1 features)

    def head_params(self):
        return list(self.proj.parameters()) + ([self.prompt] if self.prompt is not None else [])

    def passes(self):
        return [blk for _ in range(self.lm.cfg.n_repeats) for blk in self.lm.blocks]      # the unique blocks run n_repeats times

    def set_top_blocks(self, top):
        """Only the last `top` of the block passes (8 = 4 unique blocks x 2 repeats) get gradients; the embedding and the earlier
        passes run under no_grad. The unique blocks are shared by all passes, so their weights still train through the later passes."""
        self.n_nograd = max(0, len(self.passes()) - top)
        for p in self.lm.tok_emb.parameters(): p.requires_grad = False

    def hidden(self, x):
        lm, nf, ps = self.lm, self.n_nograd, self.passes()
        def embed(x):
            h = lm.tok_emb(x)
            if self.prompt is not None:
                h = torch.cat([self.prompt.unsqueeze(0).expand(x.size(0), -1, -1).to(h.dtype), h], 1)
            return h
        if nf:
            with torch.no_grad():
                h = embed(x)
                for blk in ps[:nf]: h = blk(h)
        else:
            h = embed(x)
        for blk in ps[nf:]:
            h = checkpoint(blk, h, use_reentrant=False) if (self.use_ckpt and self.training and torch.is_grad_enabled()) else blk(h)
        return lm.ln_f(h)[:, self.n_prompt:]

    def chunk_vectors(self, X, lens):
        h = self.hidden(X)
        lens = lens.to(h.device)
        m = (torch.arange(h.size(1), device=h.device)[None, :] < lens[:, None]).to(h.dtype)
        mean = (h * m.unsqueeze(-1)).sum(1) / lens.unsqueeze(-1).to(h.dtype)
        last = h[torch.arange(h.size(0), device=h.device), lens - 1]
        v = {"mean": mean, "last": last, "meanlast": torch.cat([mean, last], -1)}[self.pool]
        return v if self.raw else self.proj(v)


def build_model(method, pool="mean", r=16, alpha=32, lorma_r=8, lorma_alpha=8, n_prompt=20, bottleneck=64):
    lm = load_lm()
    prompt = 0
    if method not in ("full", "partial"):
        for p in lm.parameters(): p.requires_grad = False
    if method == "lora":        apply_lora(lm.blocks, r, alpha, rs=False)
    elif method == "rslora":    apply_lora(lm.blocks, r, alpha, rs=True)
    elif method == "lorma":     apply_lorma(lm.blocks, ("q_proj", "v_proj"), lorma_r, lorma_alpha)   # only the 4 attention blocks have q/v
    elif method == "prompt":    prompt = n_prompt
    elif method == "houlsby":   apply_adapters(lm, bottleneck, "houlsby")
    elif method == "pfeiffer":  apply_adapters(lm, bottleneck, "pfeiffer")
    elif method not in ("full", "partial"):
        raise ValueError(method)
    m = Retriever(lm, pool=pool, n_prompt=prompt).to(dev)
    if method == "partial":
        m.set_top_blocks(len(m.passes()) // 2)       # layer-wise: embedding + the first repeat (4 of 8 passes) get no gradients
    return m


def make_student(teacher):
    """Smaller student: keep unique blocks 0 and 2 (half the block parameters), still run them n_repeats times (depth 8 -> 4).
    Shared embeddings, initialised from the teacher incl. its projection. The teacher must be the plain full-fine-tuned model."""
    st = copy.deepcopy(teacher)
    keep = list(range(0, len(st.lm.blocks), 2))
    st.lm.blocks = nn.ModuleList([st.lm.blocks[i] for i in keep])
    st.lm.cfg.n_unique_blocks = len(keep)
    st.n_nograd = 0
    for p in st.parameters(): p.requires_grad = True
    return st


def n_params(m, trainable=False):
    return sum(p.numel() for p in m.parameters() if (p.requires_grad or not trainable))

## 9. Encoding, similarity functions (cosine / max-sim) and dev evaluation

In [ ]:
def encode_train(model, docs, K=None, select=None):
    """Differentiable case vectors (len(docs), EMB_DIM), L2-normalised: all chunks of all cases go through the model in one batch."""
    seqs, owner = [], []
    for i, d in enumerate(docs):
        for c in doc_chunks(d, K, select):
            seqs.append(c); owner.append(i)
    X, lens = pad_batch(seqs)
    v = model.chunk_vectors(X.to(dev), lens).float()
    tot = torch.zeros(len(docs), v.size(1), device=v.device).index_add(0, torch.tensor(owner, device=v.device), v)
    cnt = torch.bincount(torch.tensor(owner, device=v.device), minlength=len(docs)).clamp(min=1).unsqueeze(1).float()
    return F.normalize(tot / cnt, dim=-1)


@torch.no_grad()
def encode_eval(model, docs, K=None, select=None, bs=8, progress=False):
    """Case vectors and per-chunk vectors for inference (no gradients, sorted by length, batches of bs chunks).
    -> dict(doc=(D,E) normalised, chunks=[(m_i,E) normalised])"""
    model.eval()
    seqs, owner = [], []
    for i, d in enumerate(docs):
        for c in doc_chunks(d, K, select):
            seqs.append(c); owner.append(i)
        if progress and (i + 1) % 500 == 0: print(f"   tokenised {i+1}/{len(docs)}")
    order = sorted(range(len(seqs)), key=lambda j: len(seqs[j]))
    vec = None
    for b in range(0, len(order), bs):
        idx = order[b:b + bs]
        X, lens = pad_batch([seqs[j] for j in idx])
        with torch.autocast("cuda", dtype=torch.float16, enabled=dev.type == "cuda"):
            v = model.chunk_vectors(X.to(dev), lens).float()
        if vec is None: vec = torch.zeros(len(seqs), v.size(1), device=dev)
        vec[torch.tensor(idx, device=dev)] = v
    own = torch.tensor(owner, device=dev)
    tot = torch.zeros(len(docs), vec.size(1), device=dev).index_add(0, own, vec)
    cnt = torch.bincount(own, minlength=len(docs)).clamp(min=1).unsqueeze(1).float()
    chunks = []
    nvec = F.normalize(vec, dim=-1)
    counts = torch.bincount(own, minlength=len(docs)).tolist()
    pos = 0
    for c in counts:
        chunks.append(nvec[pos:pos + c]); pos += c
    return dict(doc=F.normalize(tot / cnt, dim=-1), chunks=chunks)


def score_matrix(qe, ce, mode="doc"):
    """Query x candidate scores. doc: cosine of the case vectors. maxsim: for every query chunk the best-matching candidate chunk,
    averaged over the query chunks (chunk-level late interaction)."""
    if mode == "doc":
        return (qe["doc"] @ ce["doc"].T).cpu().numpy()
    C, E = len(ce["chunks"]), ce["doc"].size(1); Kmax = max(len(c) for c in ce["chunks"])
    pad = torch.zeros(C, Kmax, E, device=dev); valid = torch.zeros(C, Kmax, dtype=torch.bool, device=dev)
    for i, c in enumerate(ce["chunks"]):
        pad[i, :len(c)] = c; valid[i, :len(c)] = True
    out = np.zeros((len(qe["chunks"]), C), dtype=np.float32)
    for qi, q in enumerate(qe["chunks"]):
        sims = torch.einsum("me,cke->cmk", q, pad).masked_fill(~valid[:, None, :], -1e4)
        out[qi] = sims.max(-1).values.mean(-1).cpu().numpy()
    return out


def evaluate_model(model, queries, cands, S_tfidf=None, name="", verbose=True, K=None, select=None):
    """Encode queries + pool once; returns the score matrices and the micro-F1@7 of each similarity function."""
    qe, ce = encode_eval(model, queries, K, select), encode_eval(model, cands, K, select, progress=verbose)
    out = {"S_doc": score_matrix(qe, ce, "doc"), "S_max": score_matrix(qe, ce, "maxsim")}
    out["doc"] = evaluate_scores(out["S_doc"], queries, cands, f"{name} doc-vector cosine", verbose)
    out["maxsim"] = evaluate_scores(out["S_max"], queries, cands, f"{name} chunk max-sim", verbose)
    return out

## 10. Time-budgeted contrastive training (all methods)

In [ ]:
def mini_dev(n_q=30, n_neg=300, seed=0):
    """A small dev subset for cheap progress checks: the first n_q dev queries, all their relevant cases and n_neg random others."""
    rng = random.Random(seed); qs = dv_q[:n_q]
    keep = {j for q in qs for j in q["rel_idx"]}
    others = [j for j in range(len(dv_c)) if j not in keep]; keep |= set(rng.sample(others, min(n_neg, len(others))))
    cs = [dv_c[j] for j in sorted(keep)]
    return qs, cs
MINI_Q, MINI_C = mini_dev()


def train_fast(method, minutes=40, B=2, n_hard=1, K=2, lr_bb=3e-5, lr_head=1e-3, tau=TAU, loss="infonce", margin=0.2,
               pool="mean", teacher=None, kd_T=0.1, kd_alpha=0.5, top_blocks=None, ckpt=True, seed=0, eval_every=15,
               init_model=None, **build_kw):
    """Time-budgeted contrastive training of the dual encoder (the schedule follows elapsed time), then a full dev evaluation.
    Each step: B queries, one random relevant case per query, n_hard tf-idf hard negatives per query. Other queries' positives and all
    hard negatives are negatives for every query (in-batch + hard negatives); relevant cases of a query are never used as its negatives.
    method: full | partial | lora | rslora | lorma | houlsby | pfeiffer | prompt | kd.     loss: infonce | margin.
    K = chunks per case while training.   top_blocks=n: only the last n of the 8 block passes train.   ckpt: gradient checkpointing
    (default True: this model's windowed attention copies key/value windows, which is memory-hungry).
    KD: loss = alpha * contrastive + (1-alpha) * KL(teacher score distribution || student score distribution) over the same columns,
        both softmaxed at temperature kd_T over the (query x column) score rows."""
    random.seed(seed); torch.manual_seed(seed)
    if init_model is not None:
        model = init_model
    elif method == "kd":
        assert teacher is not None, "pass teacher=<your fully fine-tuned model>"
        teacher.eval()
        for p in teacher.parameters(): p.requires_grad = False
        model = make_student(teacher).to(dev)
    else:
        model = build_model(method, pool=pool, **build_kw)
    model.use_ckpt = ckpt
    if top_blocks is not None: model.set_top_blocks(top_blocks)
    print(f"[{method}/{model.pool}/{loss}/hard={n_hard}/K={K}/top={top_blocks}] params {n_params(model)/1e6:.1f}M | "
          f"trainable {n_params(model, True)/1e6:.2f}M | budget {minutes} min")
    head_ids = {id(p) for p in model.head_params()}
    bb = [p for p in model.parameters() if p.requires_grad and id(p) not in head_ids]
    hd = [p for p in model.parameters() if p.requires_grad and id(p) in head_ids]
    groups, base_lrs = [], []
    if bb: groups.append({"params": bb, "lr": lr_bb}); base_lrs.append(lr_bb)
    if hd: groups.append({"params": hd, "lr": lr_head}); base_lrs.append(lr_head)
    opt = torch.optim.AdamW(groups, weight_decay=0.01)
    scaler = torch.amp.GradScaler("cuda", enabled=dev.type == "cuda")

    budget, t0, step, seen, run, nrun = minutes * 60.0, time.time(), 0, 0, 0.0, 0
    last_eval = last_log = time.time(); order = []
    while time.time() - t0 < budget:
        if len(order) < B:
            order = list(range(len(tr_q))); random.shuffle(order)
        qb = [tr_q[order.pop()] for _ in range(B)]
        pos = [random.choice(q["rel_idx"]) for q in qb]
        hard = [j for q in qb for j in (random.sample(q["hard"], min(n_hard, len(q["hard"]))) if n_hard else [])]
        cols = pos + hard                                                   # candidate columns: B positives, then the hard negatives
        docs = qb + [tr_c[j] for j in cols]
        model.train()
        with torch.autocast("cuda", dtype=torch.float16, enabled=dev.type == "cuda"):
            V = encode_train(model, docs, K)
        Vq, Vc = V[:B], V[B:]
        S = Vq @ Vc.T                                                       # (B, columns) cosine
        ban = torch.zeros_like(S, dtype=torch.bool)                         # a column that is relevant to query i (other than its own positive)
        for i, q in enumerate(qb):
            rs = set(q["rel_idx"])
            for j, cj in enumerate(cols):
                if j != i and cj in rs: ban[i, j] = True
        lab = torch.arange(B, device=dev)
        if loss == "infonce":
            ce = F.cross_entropy((S / tau).masked_fill(ban, -1e4), lab)
        else:                                                               # pairwise hinge: positive must beat every negative by `margin`
            ps = S[lab, lab].unsqueeze(1); neg_mask = ~ban; neg_mask[lab, lab] = False
            ce = (F.relu(margin - ps + S) * neg_mask).sum() / neg_mask.sum().clamp(min=1)
        total = ce
        if teacher is not None:
            with torch.no_grad(), torch.autocast("cuda", dtype=torch.float16, enabled=dev.type == "cuda"):
                Vt = encode_train(teacher, docs, K)
            St = (Vt[:B] @ Vt[B:].T)
            kl = F.kl_div(F.log_softmax((S / kd_T).masked_fill(ban, -1e4), -1), F.softmax((St / kd_T).masked_fill(ban, -1e4), -1), reduction="batchmean")
            total = kd_alpha * ce + (1 - kd_alpha) * kl
        scaler.scale(total).backward()
        prog = (time.time() - t0) / budget
        mult = prog / 0.05 if prog < 0.05 else 0.5 * (1 + math.cos(math.pi * (prog - 0.05) / 0.95))
        for g, b0 in zip(opt.param_groups, base_lrs): g["lr"] = b0 * max(mult, 0.02)
        scaler.unscale_(opt)
        torch.nn.utils.clip_grad_norm_([p for g in opt.param_groups for p in g["params"]], 1.0)
        scaler.step(opt); scaler.update(); opt.zero_grad(set_to_none=True)
        step += 1; seen += B; run += ce.item(); nrun += 1
        if eval_every and time.time() - last_eval > eval_every * 60:
            te = time.time(); print(f"   -- {(te-t0)/60:5.1f} min | loss {run/max(1,nrun):.3f} | mini-dev ({len(MINI_Q)} queries, pool {len(MINI_C)}):", end="")
            evaluate_model(model, MINI_Q, MINI_C, name="", verbose=True)
            t0 += time.time() - te; last_eval = last_log = time.time(); run, nrun = 0.0, 0
        elif time.time() - last_log > 300:
            print(f"   {(time.time()-t0)/60:5.1f} min | steps {step} | queries seen {seen} ({seen/len(tr_q):.2f} epochs) | {seen/(time.time()-t0):.2f} queries/s | loss {run/max(1,nrun):.3f}")
            run, nrun, last_log = 0.0, 0, time.time()
    print(f"trained {(time.time()-t0)/60:.1f} min: {step} steps, {seen} queries = {seen/len(tr_q):.2f} epochs")
    res = evaluate_model(model, dv_q, dv_c, name=f"DEV [{method}]")
    return model, res

## 11. Submission writer\n`prediction.json` ({query_id: [7 candidate ids]}) goes at the **root** of the zip. `run_and_submit` picks the best similarity/fusion setting on dev, writes the zip and saves the score matrices for ensembling. Upload each zip to Codabench.

In [ ]:
def write_zip(S, queries, cands, tag):
    """prediction.json = {query_id: [7 candidate ids, best first]} at the root of the zip."""
    pred = rank_top(S, queries, cands)
    pool = {c["id"] for c in cands}
    assert len(pred) == len(queries) and all(len(v) == TOPK and len(set(v)) == TOPK and set(v) <= pool for v in pred.values()), "invalid prediction"
    with open(f"{OUT}/prediction.json", "w", encoding="utf-8") as f:
        json.dump(pred, f)
    name = f"{OUT}/submission_p7_{tag}.zip"
    with zipfile.ZipFile(name, "w", zipfile.ZIP_DEFLATED) as z:
        z.write(f"{OUT}/prediction.json", arcname="prediction.json")
    print("  ", name, "| queries:", len(pred))
    return name

def run_and_submit(model, tag, res_dev=None, allow_fusion=True):
    """Dev: pick the best of {doc cosine, chunk max-sim} x {model only, fused with tf-idf (weight tuned on dev)}; test: the same recipe.
    Saves dev/test score matrices (.npy) for ensembling. Fusion with tf-idf is a ranking post-processing step - report it as such."""
    rd = res_dev or evaluate_model(model, dv_q, dv_c, name=f"DEV [{tag}]", verbose=False)
    best = None
    for mode in ("doc", "maxsim"):
        Sd = rd["S_doc" if mode == "doc" else "S_max"]
        cand = [(rd[mode]["f1"], mode, None)]
        if allow_fusion:
            for w in (0.3, 0.5, 0.7, 0.85):
                cand.append((evaluate_scores(fuse(Sd, S_tf_dv, w), dv_q, dv_c, verbose=False)["f1"], mode, w))
        for c in cand:
            if best is None or c[0] > best[0]: best = c
        print(f"[{tag}] dev {mode:6s} model-only {cand[0][0]:.4f}" + (f" | best fused {max(cand[1:])[0]:.4f} (w={max(cand[1:])[2]})" if allow_fusion else ""))
    f1, mode, w = best
    print(f"[{tag}] chosen on dev: {mode}" + (f" fused with tf-idf, w={w}" if w else " model only") + f" -> micro-F1@7 {f1:.4f}")
    qe, ce = encode_eval(model, te_q), encode_eval(model, te_c, progress=True)
    St = {"doc": score_matrix(qe, ce, "doc"), "maxsim": score_matrix(qe, ce, "maxsim")}
    np.save(f"{OUT}/p7_scores_dev_{tag}.npy", np.stack([rd["S_doc"], rd["S_max"]])); np.save(f"{OUT}/p7_scores_test_{tag}.npy", np.stack([St["doc"], St["maxsim"]]))
    S_final = fuse(St[mode], S_tf_te, w) if w else St[mode]
    write_zip(S_final, te_q, te_c, tag)
    return dict(f1=f1, mode=mode, w=w)

results = {}

## 12. Reference points: tf-idf baseline and zero-shot Phase-1 features
How good is a purely lexical ranking, and how much do the raw Phase-1 lightweight model features (mean-pooled, no training, no projection) already know about legal relevance? Both go into the transfer analysis. The tf-idf baseline is also submitted so you have one safe submission early.

In [ ]:
write_zip(S_tf_te, te_q, te_c, "tfidf_baseline")
m0 = build_model("full"); m0.raw = True
results["zero_shot"] = evaluate_model(m0, dv_q, dv_c, name="DEV zero-shot Phase-1")
del m0; torch.cuda.empty_cache()

## 13. Experiment 1 - full fine-tuning (also the KD teacher)
All weights trainable (backbone lr 3e-5, projection lr 1e-3), InfoNCE with in-batch + 1 hard negative per query, for a fixed 60-minute budget. Keep this model in memory for distillation. Memory-safe defaults (B=2 queries, gradient checkpointing) are used; if you still run out of GPU memory, set `K=1` or lower `encode_eval`'s `bs`.

In [ ]:
m_full, results["full"] = train_fast("full", minutes=60)
torch.save(m_full.state_dict(), f"{OUT}/p7_lw_full.pt")
results["full_sub"] = run_and_submit(m_full, "lw_full", results["full"])

## 14. Experiment 2 - partial (layer-wise) fine-tuning
Only the last 4 of the 8 block passes (the second repeat) and the projection train; the embedding and the first repeat run without gradients. (Blocks are shared across repeats, so their weights still receive gradient through the second repeat.)

In [ ]:
m_part, results["partial"] = train_fast("full", minutes=30, top_blocks=4)
results["partial_sub"] = run_and_submit(m_part, "lw_partial", results["partial"])

## 15. Experiment 3 - LoRA
Frozen backbone, rank-16 adapters on every Linear layer of the 4 unique blocks (attention projections and SwiGLU FFN); the projection head trains.

In [ ]:
m_lora, results["lora"] = train_fast("lora", minutes=30, lr_bb=5e-4)
results["lora_sub"] = run_and_submit(m_lora, "lw_lora", results["lora"])

## 16. Experiment 4 - LoRMA+ (compulsory)
Multiplicative low-rank adaptation (Bihany, Patel, Modi 2025) of the query/value projections of the 4 unique blocks; the projection head trains.

In [ ]:
m_lorma, results["lorma"] = train_fast("lorma", minutes=30, lr_bb=3e-4)
results["lorma_sub"] = run_and_submit(m_lorma, "lw_lorma", results["lorma"])

## 17. Experiment 5 - bottleneck adapters
Houlsby adapters (zero-initialised down/up bottlenecks after attention and after the FFN of each unique block); the backbone stays frozen. For Pfeiffer-style (FFN only) use `"pfeiffer"`.

In [ ]:
m_adapt, results["adapters"] = train_fast("houlsby", minutes=30, lr_bb=1e-3)
results["adapters_sub"] = run_and_submit(m_adapt, "lw_adapters", results["adapters"])

## 18. Experiment 6 - prompt tuning
Only 20 learned soft-prompt vectors and the projection train; the whole Phase-1 model is frozen.

In [ ]:
m_prompt, results["prompt"] = train_fast("prompt", minutes=30, lr_head=3e-3)
results["prompt_sub"] = run_and_submit(m_prompt, "lw_prompt", results["prompt"])

## 19. Experiment 7 - knowledge distillation
Teacher = experiment 1. Student = teacher with unique blocks 1 and 3 removed (blocks 0 and 2 kept, each still run twice: depth 8 -> 4), initialised from the teacher, trained on the contrastive loss plus the teacher's softened query-to-candidate score distribution. Compare parameters, encoding time and micro-F1@7 with the teacher.

In [ ]:
m_kd, results["kd"] = train_fast("kd", minutes=30, teacher=m_full, lr_bb=5e-5)
results["kd_sub"] = run_and_submit(m_kd, "lw_kd", results["kd"])
print("params: teacher", round(n_params(m_full)/1e6, 1), "M | student", round(n_params(m_kd)/1e6, 1), "M")
for nm, m in [("teacher", m_full), ("student", m_kd)]:
    torch.cuda.synchronize(); t0 = time.time(); encode_eval(m, dv_c[:200]); torch.cuda.synchronize()
    print(f"{nm}: encoding 200 candidate cases takes {time.time()-t0:.1f}s")

## 20. Pooling, loss and hard-negative ablations (optional; 20-minute top-4-block runs, compare with each other)
Pooling (`mean` vs `last` vs `meanlast`), the ranking objective (InfoNCE vs pairwise hinge) and the negative-sampling strategy (no hard negatives vs tf-idf hard negatives). Each run changes one thing relative to the first (`mean` / `infonce` / 1 hard negative).

In [ ]:
abl = {"base": dict(), "pool_last": dict(pool="last"), "pool_meanlast": dict(pool="meanlast"),
       "loss_margin": dict(loss="margin"), "no_hard_neg": dict(n_hard=0), "hard_neg_3": dict(n_hard=3)}
for name, kw in abl.items():
    m_tmp, results[f"abl_{name}"] = train_fast("full", minutes=20, top_blocks=4, eval_every=None, **kw)
    del m_tmp; torch.cuda.empty_cache()

## 21. Similarity-function and fusion ablation (inference only, experiment 1)
Case-vector cosine vs chunk max-sim, the effect of the number and placement of chunks, and fusion with tf-idf.

In [ ]:
rf = results["full"]
for w in (1.0, 0.85, 0.7, 0.5, 0.3, 0.0):
    evaluate_scores(fuse(rf["S_doc"], S_tf_dv, w), dv_q, dv_c, f"doc-cosine + tf-idf, w_model={w}")
for K_, sel in [(2, "spread"), (4, "head"), (4, "headtail"), (8, "spread")]:
    evaluate_model(m_full, dv_q, dv_c, name=f"K={K_} {sel}", K=K_, select=sel)

## 22. Ensemble (optional, usually the biggest single gain)
Averages the (row z-scored) dev score matrices of several runs - of this notebook and of the other architectures' notebooks (copy their `p7_scores_*.npy` files into `OUT`; all notebooks list queries and candidates in the same file order) - and writes a submission.

In [ ]:
tags = ["lw_full", "lw_partial", "lw_lora", "lw_lorma", "lw_adapters"]      # add tags from other notebooks
mode_idx = 0                                                                                      # 0 = case-vector cosine, 1 = chunk max-sim
D_ = [zrow(np.load(f"{OUT}/p7_scores_dev_{t}.npy")[mode_idx]) for t in tags]; T_ = [zrow(np.load(f"{OUT}/p7_scores_test_{t}.npy")[mode_idx]) for t in tags]
for t, d in zip(tags, D_): evaluate_scores(d, dv_q, dv_c, t)
ens_d, ens_t = np.mean(D_, 0), np.mean(T_, 0)
evaluate_scores(ens_d, dv_q, dv_c, "ENSEMBLE")
best_w, best_f = max(((w, evaluate_scores(w * ens_d + (1 - w) * zrow(S_tf_dv), dv_q, dv_c, verbose=False)["f1"]) for w in (1.0, 0.85, 0.7, 0.5)), key=lambda x: x[1])
print(f"ensemble + tf-idf (w={best_w}): micro-F1@7 {best_f:.4f}")
write_zip(best_w * ens_t + (1 - best_w) * zrow(S_tf_te), te_q, te_c, "ensemble")

## 23. Summary and error analysis starters

In [ ]:
print("dev micro-F1@7 (case-vector cosine / chunk max-sim):")
print(f"  {'tf-idf baseline':28s} {res_tfidf['f1']:.4f}")
for k, v in results.items():
    if isinstance(v, dict) and "doc" in v and isinstance(v["doc"], dict): print(f"  {k:28s} {v['doc']['f1']:.4f} / {v['maxsim']['f1']:.4f}")

rf = results["full"]; top = rank_top(rf["S_doc"], dv_q, dv_c)
miss = Counter(); hit_by_len = []
for q in dv_q:
    g = set(q["rel"]); h = set(top[q["id"]]) & g
    hit_by_len.append((len(q["text"]), len(h) / max(1, min(7, len(g)))))
    for r in g - set(top[q["id"]]): miss[r] += 1
print("queries with no relevant case in the top 7:", sum(1 for _, h in hit_by_len if h == 0), "of", len(dv_q))
print("gold cases missed most often:", miss.most_common(5))
L = np.array([l for l, _ in hit_by_len]); Hh = np.array([h for _, h in hit_by_len])
for lo, hi in [(0, 20000), (20000, 60000), (60000, 10**9)]:
    m = (L >= lo) & (L < hi)
    if m.any(): print(f"  query length {lo}-{hi if hi < 10**9 else '+'} chars: {int(m.sum())} queries, mean hit share {Hh[m].mean():.3f}")

## 24. (Optional) retrain on train + dev
The dev labels may be used for training once the method is chosen: append the dev queries and pool to the training data and retrain with the same budget, then submit. (Dev can then no longer be used for model selection.)